# Extra Points — Romberg Integration

## Rocket Problem

The vertical distance traveled by a rocket between $t=8$ s and $t=30$ s is given by

$$
x=\int_{8}^{30}
\left(
2000\ln\left[\frac{140000}{140000-2100t}\right]-9.8t
\right)dt.
$$

The objective is to approximate this integral using **Romberg Integration** and compare the numerical result with the exact value of the integral.

In [1]:
import numpy as np

def f(t):
  return 2000*np.log(140000/(140000-2100*t))-9.8*t

a = 8
b = 30

## Romberg Integration

Romberg Integration is a numerical integration method that improves the accuracy of the Composite Trapezoidal Rule using Richardson extrapolation.

The first approximation is obtained using the trapezoidal rule:

$$
R_{0,0}
=
\frac{b-a}{2}
\left[f(a)+f(b)\right].
$$

For the following levels, the number of subintervals is doubled:

$$
n=2^i,
$$

with step size

$$
h=\frac{b-a}{2^i}.
$$

The first column of the Romberg table is calculated using the Composite Trapezoidal Rule:

$$
R_{i,0}
=
\frac{1}{2}R_{i-1,0}
+
h
\sum_{k=1}^{2^{i-1}}
f\left(a+(2k-1)h\right).
$$

In [2]:
def trapecio_romberg(f,a,b,n):

  h = (b-a)/n
  suma = 0

  for i in range(1,n):
    suma = suma + f(a+i*h)

  integral = (h/2)*(f(a)+2*suma+f(b))

  return integral

The values obtained with the trapezoidal rule still contain numerical integration error. Romberg's method improves these approximations using Richardson extrapolation.

For each new column of the Romberg table,

$$
R_{i,j}
=
R_{i,j-1}
+
\frac{
R_{i,j-1}-R_{i-1,j-1}
}{
4^j-1
}.
$$

Each extrapolation eliminates part of the dominant truncation error of the trapezoidal approximation.

The final Romberg estimate is obtained from the last diagonal element of the table.

In [3]:
niveles = 5

R = np.zeros((niveles,niveles))

for i in range(niveles):

  n = 2**i
  R[i,0] = trapecio_romberg(f,a,b,n)

  for j in range(1,i+1):

    R[i,j] = R[i,j-1] + (R[i,j-1]-R[i-1,j-1])/(4**j-1)

In [4]:
print(R)

[[11868.34818984     0.             0.             0.
      0.        ]
 [11266.37429326 11065.71632773     0.             0.
      0.        ]
 [11112.82067637 11061.63613741 11061.36412472     0.
      0.        ]
 [11074.22129766 11061.35483809 11061.3360848  11061.33563972
      0.        ]
 [11064.55788699 11061.3367501  11061.33554424 11061.33553566
  11061.33553525]]


## Romberg Table

The Romberg table is triangular. The first column contains Composite Trapezoidal Rule approximations with an increasing number of subintervals.

The following columns contain progressively improved approximations obtained through Richardson extrapolation.

The best approximation for the selected number of levels is

$$
I_{\mathrm{Romberg}}=R_{m-1,m-1},
$$

where $m$ is the number of levels used.

In [5]:
integral_romberg = R[niveles-1,niveles-1]

print("Romberg approximation:",integral_romberg)

Romberg approximation: 11061.335535249564


## Error Analysis

To evaluate the accuracy of the method, the Romberg approximation can be compared with the exact value of the rocket integral.

The absolute error is defined as

$$
E_{\mathrm{abs}}
=
\left|
I_{\mathrm{exact}}-I_{\mathrm{Romberg}}
\right|,
$$

while the relative error is

$$
E_{\mathrm{rel}}
=
\frac{
\left|I_{\mathrm{exact}}-I_{\mathrm{Romberg}}\right|
}{
\left|I_{\mathrm{exact}}\right|
}
\times100.
$$

In [6]:
valor_exacto = 11061.335535080995

error_absoluto = abs(valor_exacto-integral_romberg)
error_relativo = (error_absoluto/abs(valor_exacto))*100

print("Exact value:",valor_exacto)
print("Romberg approximation:",integral_romberg)
print("Absolute error:",error_absoluto)
print("Relative error:",error_relativo,"%")

Exact value: 11061.335535080994
Romberg approximation: 11061.335535249564
Absolute error: 1.685693860054016e-07
Relative error: 1.5239514746730562e-09 %


## Convergence

One of the main advantages of Romberg Integration is that the successive diagonal elements of the Romberg table provide increasingly refined approximations of the integral.

The sequence

$$
R_{0,0},\quad
R_{1,1},\quad
R_{2,2},\quad
\dots
$$

can therefore be used to observe the convergence of the numerical method toward the exact value.

In [7]:
for i in range(niveles):

  aproximacion = R[i,i]
  error = abs(valor_exacto-aproximacion)

  print("Level:",i,
        "Approximation:",aproximacion,
        "Absolute error:",error)

Level: 0 Approximation: 11868.34818984112 Absolute error: 807.0126547601249
Level: 1 Approximation: 11065.716327732165 Absolute error: 4.38079265117085
Level: 2 Approximation: 11061.36412471751 Absolute error: 0.028589636514880112
Level: 3 Approximation: 11061.335639724586 Absolute error: 0.00010464359183970373
Level: 4 Approximation: 11061.335535249564 Absolute error: 1.685693860054016e-07


## Comparison with Simpson's 1/3 Rule

For the original problem, the Composite Simpson's 1/3 Rule with $n=4$ is

$$
I_S
=
\frac{h}{3}
\left[
f(t_0)
+4f(t_1)
+2f(t_2)
+4f(t_3)
+f(t_4)
\right],
$$

where

$$
h=\frac{30-8}{4}.
$$

Comparing both methods allows us to evaluate the accuracy obtained with Romberg Integration relative to the method originally used for the rocket problem.

In [8]:
n = 4
h = (b-a)/n

integral_simpson = (h/3)*(f(a)
                          + 4*f(a+h)
                          + 2*f(a+2*h)
                          + 4*f(a+3*h)
                          + f(b))

error_simpson = abs(valor_exacto-integral_simpson)
error_romberg = abs(valor_exacto-integral_romberg)

print("Simpson approximation:",integral_simpson)
print("Simpson absolute error:",error_simpson)

print("Romberg approximation:",integral_romberg)
print("Romberg absolute error:",error_romberg)

Simpson approximation: 11061.636137405923
Simpson absolute error: 0.30060232492905925
Romberg approximation: 11061.335535249564
Romberg absolute error: 1.685693860054016e-07


## Conclusion

Romberg Integration was applied to estimate the vertical distance traveled by the rocket between $t=8$ s and $t=30$ s.

The method starts from successive Composite Trapezoidal Rule approximations and improves them using Richardson extrapolation. As the Romberg table is constructed, the diagonal elements provide increasingly refined estimates of the integral.

The numerical result was compared with the exact value of the rocket integral, allowing the absolute and relative errors to be calculated. The result was also compared with the Composite Simpson's 1/3 Rule used previously.

This example shows how Romberg Integration can significantly improve a basic trapezoidal approximation without requiring a very large number of subintervals.